In [1]:
import geohash2
import warnings
import os
import time
import numpy as np
import pandas as pd
from haversine import haversine
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
veri = pd.read_csv("C:/Users/kerem/Desktop/Akıllı Trafik/Modeller/proje.csv")

veri=veri.copy()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda  x: '%.3f' % x)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

In [3]:
# ----------------- USER HYPERPARAMS (adjust) -----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

HIST = 35           # geçmiş penceresi (periyot cinsinden; 1g=5 periyot ise 7g=35)
HORIZON = 3         # tahmin penceresi
BATCH_SIZE = 16
EPOCHS = 40
LR = 1e-3
PATIENCE = 6
K_NEIGHBORS = 3

OUT_DIR = "stgcn_clean_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("Device:", DEVICE)

# ----------------- SAFE HELPERS -----------------
def safe_print_head(veri, n=5):
    print(veri.head(n).to_string())

Device: cuda


In [4]:
# ----------------- DATA CLEANING & PREP -----------------
def clean_and_prepare_dataframe(veri,
                                time_col='TIME_PERIOD',
                                time_map=None,
                                required_cols=['DATE','GEOHASH_SHORT','DENSITY']):
    """
    - Normalize TIME_PERIOD strings, map to integer period_code
    - Remove duplicate period_code columns if exist
    - Ensure DATE is datetime, drop rows missing required columns
    - Encode rain (rain_encoded) if rainfall_condition present
    - Return cleaned df
    """
    veri = veri.copy()
    # Drop exact duplicate column names (e.g., period_code, period_code_x)
    # Keep only one column named 'period_code' if present
    period_cols = [c for c in veri.columns if c.lower().startswith('period_code')]
    if len(period_cols) > 1:
        # drop all but first
        to_drop = period_cols[1:]
        veri = veri.drop(columns=to_drop)
        print(f"Dropped duplicate period_code cols: {to_drop}")

    # Ensure required columns exist
    for c in required_cols:
        if c not in veri.columns:
            raise ValueError(f"Required column missing: {c}")

    # DATE -> datetime
    veri['DATE'] = pd.to_datetime(veri['DATE'], errors='coerce')
    veri = veri.dropna(subset=['DATE','GEOHASH_SHORT','DENSITY']).reset_index(drop=True)

    # TIME_PERIOD normalization: make string, strip, replace spaces/dashes -> underscores, lowercase
    if time_col in veri.columns:
        veri[time_col] = veri[time_col].astype(str).str.strip().str.replace(r'[\s\-]+', '_', regex=True).str.lower()
    else:
        # if not present, attempt to derive from hour if exists
        if 'hour' in veri.columns:
            # example mapping; user should adapt if needed
            bins = [0,6,10,15,19,24]
            labels = ['night','morning_peak','daytime','evening_peak','late_evening']
            veri['hour'] = veri['hour'].astype(int)
            veri['TIME_PERIOD'] = pd.cut(veri['hour'], bins=bins, labels=labels, include_lowest=True).astype(str).str.lower()
            veri['TIME_PERIOD'] = veri['TIME_PERIOD'].fillna('night')
        else:
            veri['TIME_PERIOD'] = 'night'
            print("Warning: TIME_PERIOD not found; defaulting to 'night'")

    # Build mapping (normalize provided time_map to lowercase keys)
    if time_map is None:
        time_map1 = {
           'night':0,
           'morning_peak':1,
           'day_time':2,
           'evening_peak':3,
           'late_evening':4
        }
        time_map = time_map1
    else:
        # normalize keys to same normalization as TIME_PERIOD strings
        time_map = {k.strip().lower().replace(' ','_').replace('-','_'): v for k,v in time_map.items()}

    # map
    veri['period_code'] = veri['TIME_PERIOD'].map(time_map)

    # Report unmatched TIME_PERIOD values
    if veri['period_code'].isna().any():
        print("Warning: Some TIME_PERIOD values did not match mapping. Value counts (top 20):")
        print(veri.loc[veri['period_code'].isna(), 'TIME_PERIOD'].value_counts().head(20))
        # attempt fallback: use category codes deterministically
        veri['period_code'] = veri['period_code'].fillna(-1)
        # convert any -1 to category code by unique ordering
        mask = veri['period_code'] == -1
        if mask.any():
            # assign new code based on category codes but keep them consistent
            fallback_codes = pd.Categorical(veri.loc[mask, 'TIME_PERIOD']).codes
            # map fallback codes into upper range to avoid colliding with mapped values
            max_existing = max([v for v in time_map.values()]) if len(time_map)>0 else 0
            veri.loc[mask, 'period_code'] = fallback_codes + max_existing + 1
            print("Applied fallback categorical codes to unmatched TIME_PERIOD values.")
    # ensure integer
    veri['period_code'] = veri['period_code'].astype(int)

    # rainfall -> encoded
    if 'rain_encoded' not in veri.columns and 'rainfall_condition' in veri.columns:
        veri['rain_encoded'] = veri['rainfall_condition'].astype('category').cat.codes

    # other optional cols: ensure numeric and fill small gaps
    optional_num = ['MINIMUM_SPEED','MAXIMUM_SPEED','AVERAGE_SPEED','NUMBER_OF_VEHICLES','temp']
    for c in optional_num:
      if c in veri.columns:
          veri[c] = pd.to_numeric(veri[c], errors='coerce')
          veri[c] = veri.groupby("GEOHASH_SHORT")[c].transform(lambda s: s.ffill().bfill())
          veri[c] = veri[c].fillna(veri[c].median())

    return veri

In [5]:
# ----------------- ADJACENCY & NODES -----------------
def build_node_list_and_coords(veri, nodes_df=None, geohash_col='GEOHASH_SHORT'):
    if nodes_df is not None:
        nodes_unique = nodes_df.drop_duplicates(subset=[geohash_col]).reset_index(drop=True)
        node_list = nodes_unique[geohash_col].tolist()
        coords = nodes_unique[['lat','lon']].values.astype(float)
    else:
        if 'lat' in veri.columns and 'lon' in veri.columns:
            nodes_unique = veri[[geohash_col,'lat','lon']].drop_duplicates(subset=[geohash_col]).reset_index(drop=True)
            node_list = nodes_unique[geohash_col].tolist()
            coords = nodes_unique[['lat','lon']].values.astype(float)
        else:
            # decode from geohash if possible
            node_list = sorted(veri[geohash_col].drop_duplicates().tolist())
            lat_list, lon_list = [], []
            for g in node_list:
                try:
                    lat, lon = geohash2.decode(g)
                except Exception:
                    lat, lon = (0.0, 0.0)
                lat_list.append(lat); lon_list.append(lon)
            coords = np.stack([lat_list, lon_list], axis=1)
    return node_list, coords

def build_adjacency_knn(coords, k=8, sym=True):
    N = coords.shape[0]
    dist = np.zeros((N,N), dtype=float)
    for i in range(N):
        for j in range(i+1, N):
            d = haversine((coords[i,0], coords[i,1]), (coords[j,0], coords[j,1]))
            dist[i,j] = d; dist[j,i] = d
    A = np.zeros((N,N), dtype=float)
    for i in range(N):
        idx = np.argsort(dist[i])[1:k+1]
        A[i, idx] = 1.0
    if sym:
        A = np.maximum(A, A.T)
    np.fill_diagonal(A, 0.0)
    return A, dist

def normalize_adjacency(A):
    A = A.copy()
    N = A.shape[0]
    A = A + np.eye(N)
    deg = np.sum(A, axis=1)
    d_inv_sqrt = np.power(deg, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    D_inv_sqrt = np.diag(d_inv_sqrt)
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return A_norm

In [6]:
# ----------------- BUILD FEATURE TENSOR (T,N,C) -----------------
def pivot_feature_matrix(veri, node_list, feature):
    piv = veri.pivot_table(index=['DATE','period_code'],
                         columns='GEOHASH_SHORT',
                         values=feature)

    piv = piv.reindex(columns=node_list)

    # index fix (explicit MultiIndex)
    piv = piv.sort_index()
    piv = piv.reindex(index=piv.index, fill_value=np.nan)

    return piv.fillna(method='ffill').fillna(method='bfill').fillna(0.0)

def build_feature_tensor(veri, node_list, features):
    mats = []
    index_ref = None

    for feat in features:
        piv = pivot_feature_matrix(veri, node_list, feat)

        if index_ref is None:
            index_ref = piv.index
        else:
            # force MultiIndex alignment
            idx_mi = pd.MultiIndex.from_tuples(index_ref)
            piv = piv.reindex(index=idx_mi)

        mats.append(piv.values.astype(np.float32))

    data = np.stack(mats, axis=-1)  # (T, N, C)
    return data, index_ref

In [7]:
# ----------------- SEQUENCE CREATION -----------------
def create_sequences_multi(data, hist, horizon, stride=1):
    T, N, C = data.shape
    X, Y = [], []
    for t in range(0, T - hist - horizon + 1, stride):
        x = data[t:t+hist]  # (hist, N, C)
        y = data[t+hist:t+hist+horizon, :, 0]  # density channel 0
        X.append(x); Y.append(y)
    X = np.stack(X).astype(np.float32) if len(X)>0 else np.zeros((0,hist,N,C), dtype=np.float32)
    Y = np.stack(Y).astype(np.float32) if len(Y)>0 else np.zeros((0,horizon,N), dtype=np.float32)
    return X, Y

In [8]:
# ----------------- STGCN MODEL -----------------
class TemporalConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(1,kernel_size),
                              padding=(0,pad), dilation=(1,dilation))
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_channels)
    def forward(self, x):
        out = self.conv(x)
        out = self.relu(out)
        out = self.bn(out)
        return out

class GraphConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, bias=True):
        super().__init__()
        self.fc = nn.Linear(in_channels, out_channels, bias=bias)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_channels)
    def forward(self, x, A_norm):
        B, C, N, T = x.shape
        xt = x.permute(0,3,2,1).contiguous()  # (B, T, N, C)
        out_list = []
        A = A_norm.to(x.device)
        for b in range(B):
            xbt = xt[b]  # (T, N, C)
            out_bt = []
            for t in range(T):
                xtt = xbt[t]   # (N, C)
                axt = A @ xtt  # (N, C)
                out_bt.append(axt)
            out_bt = torch.stack(out_bt, dim=0)  # (T, N, C)
            out_list.append(out_bt)
        out = torch.stack(out_list, dim=0)  # (B, T, N, C)
        out = self.fc(out)  # (B, T, N, outC)
        out = out.permute(0,3,2,1).contiguous()  # (B, outC, N, T)
        out = self.relu(out)
        out = self.bn(out)
        return out

class STGCNBlock(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch, kernel_size=3):
        super().__init__()
        self.t1 = TemporalConvLayer(in_ch, mid_ch, kernel_size=kernel_size)
        self.gcn = GraphConvLayer(mid_ch, mid_ch)
        self.t2 = TemporalConvLayer(mid_ch, out_ch, kernel_size=kernel_size)
        if in_ch == out_ch:
            self.residual = lambda x: x
        else:
            self.residual = nn.Conv2d(in_ch, out_ch, kernel_size=(1,1))
        self.relu = nn.ReLU()
    def forward(self, x, A_norm):
        res = self.residual(x)
        x = self.t1(x)
        x = self.gcn(x, A_norm)
        x = self.t2(x)
        x = x + res
        x = self.relu(x)
        return x

class STGCN_Multi(nn.Module):
    def __init__(self, num_nodes, in_channels=3, blocks_config=[(16,16,32),(32,32,64)], pred_horizon=HORIZON):
        super().__init__()
        layers = []
        cin = in_channels
        for (mid1, mid2, cout) in blocks_config:
            layers.append(STGCNBlock(cin, mid1, cout))
            cin = cout
        self.blocks = nn.ModuleList(layers)
        self.final_conv = nn.Conv2d(cin, pred_horizon, kernel_size=(1,1))
        self.pred_horizon = pred_horizon
    def forward(self, x, A_norm):
        x = x.permute(0,3,2,1).contiguous()  # (B, C, N, T)
        for blk in self.blocks:
            x = blk(x, A_norm)
        out = self.final_conv(x)  # (B, pred_horizon, N, T)
        out = out[:,:,:, -1]      # (B, pred_horizon, N)
        return out

In [9]:
# ----------------- Dataset / Train Helpers -----------------
class STGCNDatasetMulti(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def train_one_epoch(model, loader, opt, criterion, A_norm_torch):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        opt.zero_grad()
        preds = model(xb, A_norm_torch)
        loss = criterion(preds, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, A_norm_torch):
    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            preds = model(xb, A_norm_torch)
            preds_all.append(preds.cpu().numpy()); trues_all.append(yb.cpu().numpy())
    if len(preds_all)==0:
        return np.nan, np.nan
    preds = np.concatenate(preds_all, axis=0); trues = np.concatenate(trues_all, axis=0)
    mae = np.mean(np.abs(preds - trues)); rmse = np.sqrt(np.mean((preds - trues)**2))
    return mae, rmse

In [10]:
# ----------------- MASTER RUN FUNCTION -----------------
def run_stgcn_pipeline_clean(veri,
                             nodes_df=None,
                             features=['DENSITY','AVERAGE_SPEED','NUMBER_OF_VEHICLES','rain_encoded','period_code'],
                             hist=HIST, horizon=HORIZON,
                             test_ratio=0.2, batch_size=BATCH_SIZE,
                             epochs=EPOCHS, lr=LR, patience=PATIENCE,
                             k_neighbors=K_NEIGHBORS, blocks_config=None):
    # 1) clean
    veri = clean_and_prepare_dataframe(veri, time_map=None)
    # 2) node list & coords
    node_list, coords = build_node_list_and_coords(veri, nodes_df)
    N = len(node_list)
    print(f"N nodes: {N}")
    # 3) adjacency
    A, dist = build_adjacency_knn(coords, k=k_neighbors)
    A_norm = normalize_adjacency(A)
    A_norm_torch = torch.tensor(A_norm, dtype=torch.float32, device=DEVICE)
    # 4) check feature presence, build final feature list
    final_feats = [f for f in features if f in veri.columns]
    if 'DENSITY' not in final_feats:
        raise ValueError("DENSITY must be present in features")
    # append period one-hot channels if 'period_code' present
    add_period_onehot = 'period_code' in final_feats
    if add_period_onehot:
        # determine number of unique period codes
        n_periods = int(veri['period_code'].max()) + 1
        # we'll expand these as additional channels after the numeric features
    # 5) build base numeric feature tensor (without period one-hot)
    numeric_feats = [f for f in final_feats if f != 'period_code']
    if len(numeric_feats) == 0:
        raise ValueError("No numeric features found in dataframe among provided features.")
    data_num, index_ref = build_feature_tensor(veri, node_list, numeric_feats)  # (T,N,C_num)
    # 6) build period one-hot mats and stack
    if add_period_onehot:
        period_vals = index_ref.get_level_values(1).astype(int).values  # aligned period codes
        period_mats = []
        n_periods = int(veri['period_code'].max()) + 1
        for p in range(n_periods):
            vec = (period_vals == p).astype(np.float32)  # (T,)
            mat = np.tile(vec.reshape(-1,1), (1, N))  # (T,N)
            period_mats.append(mat.astype(np.float32))
        # stack existing numeric channels with period mats
        mats = [data_num[:,:,i] for i in range(data_num.shape[2])]
        mats.extend(period_mats)
        data = np.stack(mats, axis=-1)  # (T,N,C)
    else:
        data = data_num
    T = data.shape[0]
    C = data.shape[2]
    print(f"T={T}, N={N}, C={C} (channels)")

    # 7) scale per-channel
    scalers = []
    data_scaled = np.zeros_like(data, dtype=np.float32)
    for c in range(C):
        arr = data[:,:,c].reshape(-1,1)
        sc = MinMaxScaler(feature_range=(0,1))
        sc.fit(arr)
        arr_s = sc.transform(arr).reshape(T,N)
        data_scaled[:,:,c] = arr_s
        scalers.append(sc)
    print("Scaling done.")

    # 8) sequences
    X, Y = create_sequences_multi(data_scaled, hist=hist, horizon=horizon, stride=1)
    if len(X) == 0:
        raise ValueError("No sequences created. Lower HIST/HORIZON or check T length.")
    S = len(X)
    split = int((1 - test_ratio) * S)
    X_train, Y_train = X[:split], Y[:split]
    X_test, Y_test = X[split:], Y[split:]
    print("Samples:", S, "Train:", len(X_train), "Test:", len(X_test))

    # 9) dataloaders
    train_ds = STGCNDatasetMulti(X_train, Y_train)
    test_ds = STGCNDatasetMulti(X_test, Y_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # 10) model
    in_ch = data.shape[2]
    if blocks_config is None:
        blocks_config=[(32,32,64),(64,64,128),(128,128,256)]
    model = STGCN_Multi(num_nodes=N, in_channels=in_ch, blocks_config=blocks_config, pred_horizon=horizon).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()

    # 11) train loop (using test_loader as val for simplicity; swap if you want separate val)
    best_val = float('inf'); best_epoch = -1
    history = {'train_loss':[], 'val_mae':[], 'val_rmse':[]}
    for ep in range(1, epochs+1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, opt, criterion, A_norm_torch)
        val_mae, val_rmse = evaluate(model, test_loader, A_norm_torch)
        history['train_loss'].append(train_loss); history['val_mae'].append(val_mae); history['val_rmse'].append(val_rmse)
        t1 = time.time()
        print(f"Epoch {ep:02d} | train_loss {train_loss:.4f} | val_mae {val_mae:.4f} | val_rmse {val_rmse:.4f} | time {(t1-t0):.1f}s")
        if val_rmse < best_val:
            best_val = val_rmse; best_epoch = ep
            torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_stgcn_clean.pth"))
        if ep - best_epoch >= patience:
            print("Early stopping.")
            break

    # 12) final eval
    model.load_state_dict(torch.load(os.path.join(OUT_DIR, "best_stgcn_clean.pth")))
    train_mae, train_rmse = evaluate(model, train_loader, A_norm_torch)
    test_mae, test_rmse = evaluate(model, test_loader, A_norm_torch)
    print("Final -> train_mae: %.4f train_rmse: %.4f | test_mae: %.4f test_rmse: %.4f" %
          (train_mae, train_rmse, test_mae, test_rmse))

    # 13) produce preds flattened and compute final metrics
    model.eval()
    preds_all, trues_all = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            preds = model(xb, A_norm_torch).cpu().numpy()
            preds_all.append(preds); trues_all.append(yb.numpy())

    preds_all = np.concatenate(preds_all, axis=0)
    trues_all = np.concatenate(trues_all, axis=0)

    preds_flat = preds_all.reshape(-1)
    trues_flat = trues_all.reshape(-1)

    # === REAL UNITS METRICS ===
    density_idx = numeric_feats.index("DENSITY")
    density_scaler = scalers[density_idx]
    preds_inv = density_scaler.inverse_transform(preds_flat.reshape(-1,1)).reshape(-1)
    trues_inv = density_scaler.inverse_transform(trues_flat.reshape(-1,1)).reshape(-1)

    final_mae_real = mean_absolute_error(trues_inv, preds_inv)
    final_rmse_real = np.sqrt(mean_squared_error(trues_inv, preds_inv))
    final_r2_real = r2_score(trues_inv, preds_inv)

    print("\n===== REAL-SCALE METRICS (DENSITY original units) =====")
    print(f"REAL MAE: {final_mae_real:.3f} | REAL RMSE: {final_rmse_real:.3f} | REAL R2: {final_r2_real:.3f}")

    return {
        'model': model,
        'history': history,
        'node_list': node_list,
        'A': A,
        'A_norm': A_norm,
        'preds': preds_all,
        'trues': trues_all,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'scalers': scalers,
        'index_ref': index_ref
    }

In [12]:
veri['rain_encoded'] = veri['rainfall_condition'].astype('category').cat.codes
veri = clean_and_prepare_dataframe(veri) # optional pre-clean
nodes_df = veri[['GEOHASH_SHORT','lat','lon']].drop_duplicates().reset_index(drop=True)
result = run_stgcn_pipeline_clean(veri, nodes_df,
                                  features=['DENSITY','AVERAGE_SPEED','NUMBER_OF_VEHICLES','rain_encoded','period_code'],
                                  hist=HIST, horizon=HORIZON,
                                  test_ratio=0.2, batch_size=BATCH_SIZE, epochs=EPOCHS)

TIME_PERIOD
daytime    54205
Name: count, dtype: int64
Applied fallback categorical codes to unmatched TIME_PERIOD values.
TIME_PERIOD
daytime    54205
Name: count, dtype: int64
Applied fallback categorical codes to unmatched TIME_PERIOD values.
N nodes: 199
T=1368, N=199, C=10 (channels)
Scaling done.
Samples: 1331 Train: 1064 Test: 267
Epoch 01 | train_loss 0.0252 | val_mae 0.0340 | val_rmse 0.0609 | time 127.5s
Epoch 02 | train_loss 0.0028 | val_mae 0.0263 | val_rmse 0.0471 | time 120.7s
Epoch 03 | train_loss 0.0024 | val_mae 0.0245 | val_rmse 0.0449 | time 122.6s
Epoch 04 | train_loss 0.0022 | val_mae 0.0235 | val_rmse 0.0436 | time 122.6s
Epoch 05 | train_loss 0.0020 | val_mae 0.0223 | val_rmse 0.0428 | time 127.3s
Epoch 06 | train_loss 0.0020 | val_mae 0.0234 | val_rmse 0.0429 | time 125.8s
Epoch 07 | train_loss 0.0018 | val_mae 0.0215 | val_rmse 0.0412 | time 123.8s
Epoch 08 | train_loss 0.0018 | val_mae 0.0216 | val_rmse 0.0416 | time 126.3s
Epoch 09 | train_loss 0.0018 | val_m